# Analyse Exploratoire des Données (EDA)
## Projet : Prédiction d'inclusion bancaire — Afrique de l'Est

---

### Objectif
Ce notebook réalise l'analyse exploratoire du dataset **Financial Inclusion** issu des enquêtes FinScope.

Il couvre les étapes suivantes :
1. Chargement et aperçu du dataset brut
2. Analyse de la structure et des types de données
3. Détection et traitement des valeurs manquantes
4. Détection des doublons
5. Normalisation des noms de colonnes
6. Export du dataset nettoyé

---

**Source** : `data/Financial_inclusion_dataset.csv`  
**Output** : `data/financial_inclusion_clean.csv`

---
## 0. Installation des dépendances

In [ ]:
%pip install matplotlib seaborn

---
## 1. Chargement du dataset

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH  = "../data/Financial_inclusion_dataset.csv"
OUTPUT_PATH = "../data/financial_inclusion_clean.csv"

df = pd.read_csv(DATA_PATH)

print(f"Dataset chargé : {df.shape[0]} lignes × {df.shape[1]} colonnes")
df.head()

Dataset chargé : 23524 lignes × 13 colonnes


,country,year,uniqueid,Has a Bank account,Type of Location,Cell Phone Access,household_size,Respondent Age,gender_of_respondent,The relathip with head,marital_status,Level of Educuation,Type of Job
0,Kenya,2018,uniqueid_1,Yes,Rural,Yes,3.0,24.0,Female,Spouse,Married/Living together,Secondary education,Self employed
1,Kenya,2018,uniqueid_2,No,Rural,No,5.0,70.0,Female,Head of Household,Widowed,No formal education,Government Dependent
2,Kenya,2018,uniqueid_3,Yes,Urban,Yes,5.0,26.0,Male,Other relative,Single/Never Married,Vocational/Specialised training,Self employed
3,Kenya,2018,uniqueid_4,No,Rural,Yes,5.0,34.0,Female,Head of Household,Married/Living together,Primary education,Formally employed Private
4,Kenya,2018,uniqueid_5,No,Urban,No,8.0,26.0,Male,Child,Single/Never Married,Primary education,Informally employed


---
## 2. Structure et types de données

In [2]:
# Dimensions du dataset
print(f"Dimensions : {df.shape}\n")

# Types de colonnes et valeurs non-nulles
print("=== Informations générales ===")
df.info()

print("\n=== Statistiques descriptives (colonnes numériques) ===")
df.describe()

Dimensions : (23524, 13)

=== Informations générales ===
<class 'pandas.DataFrame'>
RangeIndex: 23524 entries, 0 to 23523
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   country                 23510 non-null  str    
 1   year                    23524 non-null  int64  
 2   uniqueid                23524 non-null  str    
 3   Has a Bank account      23488 non-null  str    
 4   Type of Location        23509 non-null  str    
 5   Cell Phone Access       23513 non-null  str    
 6   household_size          23496 non-null  float64
 7   Respondent Age          23490 non-null  float64
 8   gender_of_respondent    23490 non-null  str    
 9   The relathip with head  23520 non-null  str    
 10  marital_status          23492 non-null  str    
 11  Level of Educuation     23495 non-null  str    
 12  Type of Job             23494 non-null  str    
dtypes: float64(2), int64(1), str(10)
memory usage

,year,household_size,Respondent Age
count,23524.000000,23496.000000,23490.000000
mean,2016.979000,3.681818,38.804300
std,0.899669,2.279933,16.519996
min,2016.000000,0.000000,16.000000
25%,2016.000000,2.000000,26.000000
50%,2017.000000,3.000000,35.000000
75%,2018.000000,5.000000,49.000000
max,2056.000000,21.000000,100.000000


In [3]:
# Séparation colonnes numériques / catégorielles
num_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()

print(f"Colonnes numériques  ({len(num_cols)}) : {num_cols}")
print(f"Colonnes catégorielles ({len(cat_cols)}) : {cat_cols}")

Colonnes numériques  (3) : ['year', 'household_size', 'Respondent Age']
Colonnes catégorielles (10) : ['country', 'uniqueid', 'Has a Bank account', 'Type of Location', 'Cell Phone Access', 'gender_of_respondent', 'The relathip with head', 'marital_status', 'Level of Educuation', 'Type of Job']


C:\Users\marcs\AppData\Local\Temp\ipykernel_25716\2701340567.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=["object"]).columns.tolist()


---
## 3. Analyse et traitement des valeurs manquantes

In [4]:
# Détection des valeurs manquantes
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_report = pd.DataFrame({
    "Valeurs manquantes": missing,
    "Pourcentage (%)": missing_pct
}).query("`Valeurs manquantes` > 0")

if missing_report.empty:
    print("✅ Aucune valeur manquante détectée.")
else:
    print("⚠️ Colonnes avec valeurs manquantes :")
    display(missing_report)

⚠️ Colonnes avec valeurs manquantes :


,Valeurs manquantes,Pourcentage (%)
country,14,0.06
Has a Bank account,36,0.15
Type of Location,15,0.06
Cell Phone Access,11,0.05
household_size,28,0.12
Respondent Age,34,0.14
gender_of_respondent,34,0.14
The relathip with head,4,0.02
marital_status,32,0.14
Level of Educuation,29,0.12


In [5]:
# Traitement des valeurs manquantes
# - Colonnes numériques  → remplacement par la médiane (robuste aux outliers)
# - Colonnes catégorielles → remplacement par 'Unknown'

for col in num_cols:
    if df[col].isnull().sum() > 0:
        median_value = df[col].median()
        df[col].fillna(median_value, inplace=True)
        print(f"  [NUM] '{col}' → médiane = {median_value}")

for col in cat_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna("Unknown", inplace=True)
        print(f"  [CAT] '{col}' → 'Unknown'")

# Vérification finale
remaining_nulls = df.isnull().sum().sum()
print(f"\n✅ Valeurs manquantes restantes : {remaining_nulls}")

  [NUM] 'household_size' → médiane = 3.0
  [NUM] 'Respondent Age' → médiane = 35.0
  [CAT] 'country' → 'Unknown'
  [CAT] 'Has a Bank account' → 'Unknown'
  [CAT] 'Type of Location' → 'Unknown'
  [CAT] 'Cell Phone Access' → 'Unknown'
  [CAT] 'gender_of_respondent' → 'Unknown'
  [CAT] 'The relathip with head' → 'Unknown'
  [CAT] 'marital_status' → 'Unknown'
  [CAT] 'Level of Educuation' → 'Unknown'
  [CAT] 'Type of Job' → 'Unknown'

✅ Valeurs manquantes restantes : 267


C:\Users\marcs\AppData\Local\Temp\ipykernel_25716\1841574461.py:8: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df[col].fillna(median_value, inplace=True)
C:\Users\marcs\AppData\Local\Temp\ipykernel_25716\1841574461.py:8: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using

---
## 4. Détection des doublons

In [6]:
duplicate_count = df.duplicated().sum()

if duplicate_count == 0:
    print("✅ Aucun doublon détecté.")
else:
    print(f"⚠️ {duplicate_count} doublon(s) détecté(s) — suppression en cours...")
    df = df.drop_duplicates()
    print(f"✅ Dataset après suppression : {df.shape[0]} lignes")

✅ Aucun doublon détecté.


---
## 5. Normalisation des noms de colonnes

Standardisation en `snake_case` : suppression des espaces, mise en minuscules.

In [7]:
print("Colonnes avant normalisation :")
print(df.columns.tolist())

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print("\nColonnes après normalisation :")
print(df.columns.tolist())

Colonnes avant normalisation :
['country', 'year', 'uniqueid', 'Has a Bank account', 'Type of Location', 'Cell Phone Access', 'household_size', 'Respondent Age', 'gender_of_respondent', 'The relathip with head', 'marital_status', 'Level of Educuation', 'Type of Job']

Colonnes après normalisation :
['country', 'year', 'uniqueid', 'has_a_bank_account', 'type_of_location', 'cell_phone_access', 'household_size', 'respondent_age', 'gender_of_respondent', 'the_relathip_with_head', 'marital_status', 'level_of_educuation', 'type_of_job']


---
## 6. État final du dataset

In [8]:
print(f"✅ Dataset nettoyé : {df.shape[0]} lignes × {df.shape[1]} colonnes")
df.info()
df.head()

✅ Dataset nettoyé : 23524 lignes × 13 colonnes
<class 'pandas.DataFrame'>
RangeIndex: 23524 entries, 0 to 23523
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   country                 23510 non-null  str    
 1   year                    23524 non-null  int64  
 2   uniqueid                23524 non-null  str    
 3   has_a_bank_account      23488 non-null  str    
 4   type_of_location        23509 non-null  str    
 5   cell_phone_access       23513 non-null  str    
 6   household_size          23496 non-null  float64
 7   respondent_age          23490 non-null  float64
 8   gender_of_respondent    23490 non-null  str    
 9   the_relathip_with_head  23520 non-null  str    
 10  marital_status          23492 non-null  str    
 11  level_of_educuation     23495 non-null  str    
 12  type_of_job             23494 non-null  str    
dtypes: float64(2), int64(1), str(10)
memory usage: 2.3 MB


,country,year,uniqueid,has_a_bank_account,type_of_location,cell_phone_access,household_size,respondent_age,gender_of_respondent,the_relathip_with_head,marital_status,level_of_educuation,type_of_job
0,Kenya,2018,uniqueid_1,Yes,Rural,Yes,3.0,24.0,Female,Spouse,Married/Living together,Secondary education,Self employed
1,Kenya,2018,uniqueid_2,No,Rural,No,5.0,70.0,Female,Head of Household,Widowed,No formal education,Government Dependent
2,Kenya,2018,uniqueid_3,Yes,Urban,Yes,5.0,26.0,Male,Other relative,Single/Never Married,Vocational/Specialised training,Self employed
3,Kenya,2018,uniqueid_4,No,Rural,Yes,5.0,34.0,Female,Head of Household,Married/Living together,Primary education,Formally employed Private
4,Kenya,2018,uniqueid_5,No,Urban,No,8.0,26.0,Male,Child,Single/Never Married,Primary education,Informally employed


---
## 7. Export du dataset nettoyé

In [9]:
df.to_csv(OUTPUT_PATH, index=False)
print(f"✅ Dataset exporté : {OUTPUT_PATH}")

✅ Dataset exporté : ../data/financial_inclusion_clean.csv
